# PortWatch Tanker Port Calls - Cumulative Daily Data by Country
Uses existing `Daily_Ports_Data.csv` (up to 2026-02-06) and fetches missing recent data from the API to get up to the most recent available date. Outputs cumulative daily tanker port calls by country for 2024-2026.

In [1]:
import requests
import pandas as pd
import time
from datetime import date

BASE_URL = "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/Daily_Ports_Data/FeatureServer/0/query"
YEARS = [2024, 2025, 2026]

In [2]:
# Load existing daily data and filter to 2024+
existing = pd.read_csv("Daily_Ports_Data.csv", usecols=["date", "year", "month", "day", "country", "portcalls_tanker"])
existing["date"] = pd.to_datetime(existing["date"])
existing = existing[existing["year"] >= 2024].copy()

last_date = existing["date"].max()
print(f"Existing data: {existing['date'].min().date()} to {last_date.date()}")
print(f"Records (2024+): {len(existing):,}")

Existing data: 2024-01-01 to 2026-02-06
Records (2024+): 1,524,480


In [3]:
# Fetch missing data from API (after last_date in existing CSV)
# Uses server-side aggregation to get country-day level totals directly

def fetch_aggregated_page(where_clause, offset=0, page_size=1000):
    params = {
        "where": where_clause,
        "outStatistics": '[{"statisticType":"sum","onStatisticField":"portcalls_tanker","outStatisticFieldName":"portcalls_tanker"}]',
        "groupByFieldsForStatistics": "country,year,month,day",
        "resultRecordCount": page_size,
        "resultOffset": offset,
        "f": "json"
    }
    resp = requests.get(BASE_URL, params=params, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    features = data.get("features", [])
    records = [f["attributes"] for f in features]
    exceeded = data.get("exceededTransferLimit", False)
    return records, exceeded

# Build where clause for dates after last existing date
last_year = last_date.year
last_month = last_date.month
last_day = last_date.day

# Query: same year as last_date but after that month/day, plus any later year
# We query year=2026 AND (month > last_month OR (month = last_month AND day > last_day))
where = (
    f"year = {last_year} AND "
    f"(month > {last_month} OR (month = {last_month} AND day > {last_day}))"
)
print(f"Fetching new data with: {where}")

all_new = []
offset = 0
page = 0
while True:
    records, exceeded = fetch_aggregated_page(where, offset)
    all_new.extend(records)
    page += 1
    if page % 10 == 0:
        print(f"  Fetched {len(all_new)} records so far...")
    if not exceeded or len(records) == 0:
        break
    offset += len(records)
    time.sleep(0.3)

print(f"New records fetched: {len(all_new)}")
if all_new:
    new_df = pd.DataFrame(all_new)
    new_df["date"] = pd.to_datetime(new_df[["year", "month", "day"]])
    print(f"New data range: {new_df['date'].min().date()} to {new_df['date'].max().date()}")
else:
    new_df = pd.DataFrame(columns=["country", "year", "month", "day", "portcalls_tanker", "date"])

Fetching new data with: year = 2026 AND (month > 2 OR (month = 2 AND day > 6))
New records fetched: 8967
New data range: 2026-02-07 to 2026-03-27


In [4]:
# Aggregate existing data to country-day level and combine with new API data
existing_agg = (
    existing.groupby(["country", "date"], as_index=False)["portcalls_tanker"]
    .sum()
)
existing_agg["year"] = existing_agg["date"].dt.year
print(f"Existing aggregated: {len(existing_agg):,} country-day records")

# Combine existing + new
if len(new_df) > 0:
    new_subset = new_df[["country", "date", "portcalls_tanker", "year"]].copy()
    df = pd.concat([existing_agg, new_subset], ignore_index=True)
else:
    df = existing_agg.copy()

df = df.rename(columns={"portcalls_tanker": "tanker_calls"})
print(f"Combined: {len(df):,} records")
print(f"Countries: {df['country'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

Existing aggregated: 138,240 country-day records
Combined: 147,207 records
Countries: 183
Date range: 2024-01-01 to 2026-03-27


In [5]:
# Build complete country x date grid so missing dates get 0
countries = sorted(df["country"].unique())
max_date = df["date"].max().date()

frames = []
for yr in YEARS:
    start = date(yr, 1, 1)
    end = min(date(yr, 12, 31), max_date)
    if start > end:
        continue
    dates = pd.date_range(start, end, freq="D")
    for c in countries:
        tmp = pd.DataFrame({"country": c, "date": dates, "year": yr})
        frames.append(tmp)

grid = pd.concat(frames, ignore_index=True)
print(f"Grid size: {len(grid):,}")

# Merge actual data onto grid
merged = grid.merge(df[["country", "date", "tanker_calls"]], on=["country", "date"], how="left")
merged["tanker_calls"] = merged["tanker_calls"].fillna(0).astype(int)
print(f"Merged size: {len(merged):,}")

Grid size: 149,511
Merged size: 149,511


In [6]:
# Compute cumulative tanker calls per country per year
merged = merged.sort_values(["country", "year", "date"]).reset_index(drop=True)
merged["cumulative_tanker"] = merged.groupby(["country", "year"])["tanker_calls"].cumsum()
merged["day_of_year"] = merged["date"].dt.dayofyear

merged.head(10)

,country,date,year,tanker_calls,cumulative_tanker,day_of_year
0,Albania,2024-01-01,2024,0,0,1
1,Albania,2024-01-02,2024,1,1,2
2,Albania,2024-01-03,2024,0,1,3
3,Albania,2024-01-04,2024,0,1,4
4,Albania,2024-01-05,2024,0,1,5
5,Albania,2024-01-06,2024,0,1,6
6,Albania,2024-01-07,2024,0,1,7
7,Albania,2024-01-08,2024,0,1,8
8,Albania,2024-01-09,2024,0,1,9
9,Albania,2024-01-10,2024,1,2,10


In [7]:
# Pivot to final format: rows = country + day_of_year, columns = year
pivot = merged.pivot_table(
    index=["country", "day_of_year"],
    columns="year",
    values="cumulative_tanker",
    aggfunc="first"
)

pivot.columns = [str(int(c)) for c in pivot.columns]
pivot = pivot.reset_index()

print(f"Final shape: {pivot.shape}")
pivot.head(20)

Final shape: (66978, 5)


,country,day_of_year,2024,2025,2026
0,Albania,1,0.0,0.0,1.0
1,Albania,2,1.0,1.0,1.0
2,Albania,3,1.0,1.0,1.0
3,Albania,4,1.0,1.0,1.0
4,Albania,5,1.0,1.0,1.0
5,Albania,6,1.0,3.0,1.0
6,Albania,7,1.0,3.0,1.0
7,Albania,8,1.0,3.0,1.0
8,Albania,9,1.0,3.0,1.0
9,Albania,10,2.0,3.0,1.0


In [ ]:
# Save to CSV
output_path = "portwatch_tanker_cumulative_by_country.csv"
pivot.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {pivot.shape}")
print(f"\nSample (United States):")
pivot[pivot["country"] == "United States"].tail(10)

In [ ]:
pivot